# Task 8 Prompt Engineering

In [20]:
# 1) Imports
import os, json, re, time
from collections import Counter, defaultdict
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# matplotlib display tweaks (no specific colors per course rules is fine; we won't set styles)
plt.rcParams['figure.figsize'] = (7,5)


## 1) Config API token & base URL


In [21]:
# 2) Load .env / fallbacks
load_dotenv()

API_KEY  = os.getenv("LLM_API_KEY",  "YOUR_LLM_API_KEY")
API_BASE = os.getenv("LLM_API_BASE", "http://194.171.191.228:30080")  # update if needed
MODEL    = os.getenv("LLM_MODEL",    "meta-llama/Llama-3.3-70B-Instruct")

assert API_BASE.startswith("http"), "Please set LLM_API_BASE to your local server base URL."
if not API_KEY:
    print("⚠️  LLM_API_KEY not found in environment; set it in a .env file before running the API calls.")
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
print("Using base:", API_BASE, "| model:", MODEL)


Using base: http://194.171.191.228:30080 | model: meta-llama/Llama-3.3-70B-Instruct


## 2) Helper Chat client (Open WebUI/OpenAI‑compatible `/api/chat/completions`)

In [22]:

import requests, time, random

SESSION = requests.Session()

def chat_text(
    prompt,
    model=MODEL,             # define MODEL earlier, e.g. "meta-llama/Llama-3.3-70B-Instruct"
    temperature=0.0,
    max_tokens=24,
    timeout=60,
    stop=None,               # None to avoid truncating JSON early
    retries=5
):
    url = f"{API_BASE}/api/chat/completions"
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
    }
    if stop is not None:
        data["stop"] = stop

    last_err = None
    for attempt in range(retries):
        try:
            r = SESSION.post(url, headers=headers, json=data, timeout=timeout)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            last_err = e
            # exponential backoff with jitter
            sleep = min(8, 0.8 * (2 ** attempt)) + random.random() * 0.25
            if attempt == retries - 1:
                # include server text if available
                try:
                    msg = r.text[:500]
                except Exception:
                    msg = ""
                raise RuntimeError(f"LLM call failed after {retries} retries. Last error: {e}\nServer said: {msg}") from e
            time.sleep(sleep)

# --- Caching layer (AFTER chat_text is defined) ---
import shelve, hashlib, json as _json

CACHE_PATH = "llm_cache.db"  # persists in my working folder

def _cache_key(prompt, **kw):
    # include all kwargs (model/temp/max_tokens/stop/etc.) to avoid collisions
    payload = {"prompt": prompt, **kw}
    return hashlib.sha1(
        _json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def chat_text_cached(prompt, **kw):
    key = _cache_key(prompt, **kw)
    with shelve.open(CACHE_PATH) as db:
        if key in db:
            return db[key]
        out = chat_text(prompt, **kw)   # uses the retrying client
        db[key] = out
        return out

def clear_llm_cache():
    with shelve.open(CACHE_PATH) as db:
        db.clear()


## 3) Load the annotated data
Expected columns (case-insensitive accepted):  
- `Sentence` (text)  
- `Emotion_core` (gold label in {happiness, sadness, anger, surprise, fear, disgust, neutral})  


In [23]:
DATA_PATH = r"C:\Users\custo\Desktop\Task 8\group_23_url_1_transcript.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    quotechar='"',
    doublequote=True,
    parse_dates=["Start Time","End Time"],   # optional, nice to have
)

# Keep only what we need; normalize label
df = df[["Sentence", "Emotion_core", "Intensity"]].rename(columns={"Emotion_core": "Emotion"})
df["Emotion"] = df["Emotion"].str.lower().str.strip()

print(df.head(8))
print("\nLabel distribution:\n", df["Emotion"].value_counts())


                                            Sentence    Emotion Intensity
0  Previously on Australian Survivor, Tara and Lo...  happiness      mild
1  I'm trusting you with this because I'm sitting...       fear  moderate
2        But Tara decided it was time for a breakup.    sadness  moderate
3            It breaks my heart, but he's got to go.    sadness   intense
4       It wasn't the only moral dilemma Tara faced.    neutral   neutral
5                My eyes just popped out of my head.   surprise   intense
6                             I don't even feel bad.    neutral   neutral
7  Tara sweet-talked the tribe into voting with h...    neutral      mild

Label distribution:
 Emotion
neutral      209
happiness    156
fear         101
sadness       88
anger         54
surprise      51
disgust        6
Name: count, dtype: int64



## 4) Create a dev/test split
Use **dev** for prompt iteration. Keep **test** untouched for a fair final score.


In [24]:
# Stratified split (80/20) if enough samples per class, otherwise simple split.
labels = df["Emotion"].unique().tolist()
try:
    df_dev, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Emotion"])
except Exception:
    df_dev, df_test = train_test_split(df, test_size=0.2, random_state=42)

print("Dev:", df_dev.shape, " Test:", df_test.shape)

Dev: (532, 3)  Test: (133, 3)


# 5) Quick baseline to confirm end-to-end

In [ ]:
# ===== Baseline =====
ALLOWED = {"happiness","sadness","anger","surprise","fear","disgust","neutral"}
def parse_label(s):
    s = str(s).lower()
    for lab in ALLOWED:
        if lab in s:
            return lab
    return "neutral"

BASELINE_PROMPT = """Classify the following sentence into exactly one of:
[happiness, sadness, anger, surprise, fear, disgust, neutral].
Sentence: {text}
Answer ONLY with the single label.
"""

def classify_baseline(text):
    raw = chat_text_cached(BASELINE_PROMPT.format(text=text), temperature=0.0, max_tokens=16)
    return parse_label(raw)

sample_dev = df_dev.sample(50, random_state=0).copy()
sample_dev["Predicted"] = sample_dev["Sentence"].apply(classify_baseline)
print(classification_report(sample_dev["Emotion"], sample_dev["Predicted"], digits=3))
print("Macro-F1:", f1_score(sample_dev["Emotion"], sample_dev["Predicted"], average="macro"))

              precision    recall  f1-score   support

       anger      1.000     0.800     0.889         5
     disgust      0.500     1.000     0.667         1
        fear      1.000     0.571     0.727         7
   happiness      0.750     0.300     0.429        10
     neutral      0.517     1.000     0.682        15
     sadness      1.000     0.600     0.750        10
    surprise      1.000     0.500     0.667         2

    accuracy                          0.680        50
   macro avg      0.824     0.682     0.687        50
weighted avg      0.795     0.680     0.671        50

Macro-F1: 0.6871263656977942


In [26]:
# ===== Curated + topped-up few-shots (STRONGER prototypes) =====
import numpy as np, random, re
from collections import Counter

# optional: normalize curly quotes in curated text
def _norm(s: str) -> str:
    return (s.replace("\u2019", "'").replace("\u2018", "'")
             .replace("\u201c", '"').replace("\u201d", '"'))

CURATED = {
    "neutral":  [_norm(x) for x in ["Six are left.", "Member of our jury, Luke.", "It was Lockie's second straight challenge win."]],
    "anger":    [_norm(x) for x in ["God damn it!", "You've got the wrong one.", "Go!"]],
    "surprise": [_norm(x) for x in ["Survivors ready?", "I didn't expect that twist.", "What just happened?", "No way!"]],
    "happiness":[_norm(x) for x in ["Thanks.", "Nice to meet you.", "We did it!"]],
    "fear":     [_norm(x) for x in ["I'm worried they'll target me next.", "This is scary."]],
    "sadness":  [_norm(x) for x in ["It breaks my heart.", "Farewell, Lukey.", "I'm devastated.", "We lost him."]],
    "disgust":  [_norm(x) for x in ["That was disgusting.", "Makes me feel sick."]],
}

CLASSES = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

def compose_fewshots_balanced(df_dev, curated=CURATED, base_k=4, boost=2, max_len=80, seed=42):
    rng = random.Random(seed)
    freq = Counter(df_dev["Emotion"])
    med = np.median(list(freq.values())) if freq else 0

    # more examples for below-median classes; keep neutral small
    k_by = {c: (base_k + (boost if freq.get(c,0) < med else 0)) for c in CLASSES}
    k_by["neutral"] = min(k_by["neutral"], 3)

    pairs = []
    have = {c: [] for c in CLASSES}

    # curated seeds first
    for c, texts in curated.items():
        for t in texts:
            t = str(t).strip()
            if 2 <= len(t) <= max_len:
                pairs.append((t, c)); have[c].append(t)

    # top up from DEV to reach target per class
    for c in CLASSES:
        need = max(0, k_by[c] - len(have[c]))
        if need == 0: 
            continue
        candidates = (
            df_dev[df_dev["Emotion"] == c]["Sentence"]
              .dropna().astype(str).map(str.strip)
              .loc[lambda s: s.str.len().between(2, max_len)]
              .unique().tolist()
        )
        candidates = [s for s in candidates if s not in have[c]]
        if len(candidates) > need:
            candidates = rng.sample(candidates, need)
        for s in candidates:
            pairs.append((s, c)); have[c].append(s)

    # optional: randomize final order so the prompt varies
    rng.shuffle(pairs)
    return pairs

fewshot_pairs = compose_fewshots_balanced(df_dev, base_k=4, boost=2)
print("Few-shot pairs:", len(fewshot_pairs))
print("Per-class counts:", Counter(l for _, l in fewshot_pairs))

# ---- Class weights (tie-breaking / voting bias) ----
# ensure ALLOWED exists even if earlier cell didn't run yet
try:
    ALLOWED
except NameError:
    ALLOWED = {"happiness","sadness","anger","surprise","fear","disgust","neutral"}

_dev_counts = Counter(df_dev["Emotion"])
_total = sum(_dev_counts.values()) or 1

# inverse-sqrt prior; you can tune exponent 0.5 -> 0.3..0.7
weights = {}
for c in ALLOWED:
    prior = _dev_counts.get(c, 1) / _total
    weights[c] = 1.0 / max(prior, 1e-6) ** 0.5

# penalize neutral a bit so it doesn't win ties
weights["neutral"] *= 0.7

print("Class weights:", {k: round(v, 2) for k, v in weights.items()})

Few-shot pairs: 33
Per-class counts: Counter({'surprise': 6, 'anger': 6, 'disgust': 6, 'happiness': 4, 'sadness': 4, 'fear': 4, 'neutral': 3})
Class weights: {'sadness': 2.76, 'happiness': 2.06, 'fear': 2.56, 'disgust': 10.32, 'neutral': 1.25, 'anger': 3.52, 'surprise': 3.6}



## 6) Parsing & label normalization
- Force strict JSON where possible, but be robust (regex fallback).  
- Map synonyms (CIA-style) to core labels.


In [27]:
# ==== Helpers (single source of truth) ====
import re, json
from collections import Counter

ALLOWED = {"happiness","sadness","anger","surprise","fear","disgust","neutral"}
NON_NEUTRAL = {"happiness","sadness","anger","surprise","fear","disgust"}

# Parse {"label": "..."} (generic) with a safe fallback
def parse_json_or_label(s):
    s = str(s).strip()
    # Try strict JSON first
    try:
        obj = json.loads(s)
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).lower().strip()
    except Exception:
        pass
    # Fallback: try to read any known label-ish token (keeps this generic)
    m = re.search(r'([a-z]+)', s, re.I)
    return m.group(1).lower() if m else None

# High-precision lexical rules (fast, conservative)
LEX = {
    "happiness": re.compile(r"\b(thanks|thank you|nice to meet you|proud|glad|happy|win|victor(y|ies)|congrats?)\b", re.I),
    "sadness":   re.compile(r"\b(breaks my heart|farewell|gutted|sad|goodbye|loss|break ?up|heartbroken|devastated)\b", re.I),
    "anger":     re.compile(r"\b(god ?damn|you('?| )?ve got the wrong one|angry|furious|frustrat(ed|ing)|mad)\b|^go!?$", re.I),
    "surprise":  re.compile(r"\b(survivors ready\??|didn'?t expect|what just happened|twist|shocked|wow|all of a sudden|no way)\b", re.I),
    "fear":      re.compile(r"\b(too dangerous|target me|worried|afraid|scared|terrified|threat|unsafe|nervous|anxious)\b", re.I),
    "disgust":   re.compile(r"\b(disgust(ing)?|gross|feel sick|nausea|revolting|yuck|ew+w?)\b", re.I),
}

FILLER_RE = re.compile(r'^(yeah|yep|right\?|ok(ay)?|mm+|uh+|oh)\.?$', re.I)

def is_trivial_neutral(text: str) -> bool:
    t = (text or "").strip()
    return len(t) < 4 or FILLER_RE.match(t) is not None

# Rare-class boost (we check disgust first)
DISGUST_CUE = LEX["disgust"]

def rule_label(text):
    if is_trivial_neutral(text):
        return "neutral"
    if DISGUST_CUE.search(text or ""):
        return "disgust"
    for lab, rx in LEX.items():
        if lab != "disgust" and rx.search(text or ""):
            return lab
    return None

def _normalize_quotes(text: str) -> str:
    return (text or "").replace("\u2019", "'").replace("\u2018", "'") \
                       .replace("\u201c", '"').replace("\u201d", '"')

def has_emotion_cue(text):
    text = _normalize_quotes(text)
    if any(rx.search(text) for rx in LEX.values()): 
        return True
    if re.search(r"[!?\u2757]", text): 
        return True
    if re.search(r"\b(i('| )?m|i am)\b", text, re.I): 
        return True
    return False

# Super-cheap polarity cue (only for tie nudges)
POS_RE = LEX["happiness"]
NEG_RE = re.compile(r"\b(afraid|scared|terrified|dangerous|worried|target me|angry|furious|frustrat(ed|ing)|sad|gutted|farewell|disgust(ing)?|gross|sick|damn)\b", re.I)

def guess_polarity(text: str) -> str:
    if POS_RE.search(text or "") and not NEG_RE.search(text or ""):
        return "positive"
    if NEG_RE.search(text or "") and not POS_RE.search(text or ""):
        return "negative"
    return "unknown"


In [28]:
# ===== Confidence-aware parsing + scoped classifier with tau =====
import re, json
from collections import Counter

# Parse {"label": "...", "confidence": 0.xx} or fall back to best-effort
def parse_label_and_confidence(s):
    s = str(s).strip()
    lab = None
    conf = None
    # Try strict JSON
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            if "label" in obj:
                lab = str(obj["label"]).lower().strip()
            if "confidence" in obj:
                try:
                    conf = float(obj["confidence"])
                except Exception:
                    conf = None
    except Exception:
        pass
    # Fallback regex for label
    if not lab:
        m = re.search(r'(happiness|sadness|anger|surprise|fear|disgust|neutral)', s, re.I)
        if m:
            lab = m.group(1).lower()
    # Fallback regex for confidence
    if conf is None:
        m = re.search(r'"?confidence"?\s*[:=]\s*([0-9]*\.?[0-9]+)', s, re.I)
        if m:
            try:
                conf = float(m.group(1))
            except Exception:
                conf = None
    # clip to [0,1]
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    return lab, conf

# Polarity gate (positive | negative | neutral) — braces escaped
POLARITY_PROMPT = """You are classifying the overall sentiment polarity of a single sentence.
Return strict JSON only: {{ "label": "positive"|"negative"|"neutral" }}.
Sentence: {text}
JSON:
"""
POLARITY_SET = {"positive","negative","neutral"}

def parse_polarity(s: str):
    # First try JSON, then fallback regex (more robust than regex-only)
    s = str(s).strip()
    try:
        obj = json.loads(s)
        if isinstance(obj, dict) and "label" in obj:
            lab = str(obj["label"]).lower().strip()
            if lab in POLARITY_SET:
                return lab
    except Exception:
        pass
    m = re.search(r'\b(positive|negative|neutral)\b', s, re.I)
    return m.group(1).lower() if m else None

def polarity_gate(text):
    raw = chat_text_cached(POLARITY_PROMPT.format(text=text), temperature=0.0, max_tokens=24)
    lab = parse_polarity(raw)
    return lab if lab in POLARITY_SET else "neutral"

# Few-shot (scoped) builder; reuses your fewshot_pairs
def _label_block(labels):
    return "[" + ", ".join(sorted(labels)) + "]"

def make_fewshot_block_scoped(pairs, allowed, k_total=12, seed=123):
    import random
    rng = random.Random(seed)
    cand = [(s,l) for (s,l) in pairs if l in allowed]
    if not cand:  # fallback
        cand = pairs[:]
    rng.shuffle(cand)
    picked = cand[:min(k_total, len(cand))]
    return "\n".join([f'Sentence: "{s}" -> Emotion: {l}' for s,l in picked])

# Confidence-aware prompt (scoped to allowed labels)
CLASSIFY_SET_PROMPT_CONF = """You label ONE sentence with EXACTLY ONE emotion from this set: {label_set}.
Return strict JSON only: {{"label": one of {label_set}, "confidence": a float 0..1}}.
Confidence should reflect how sure you are (probability-like).
Use these examples as a guide:
{text_label_examples}

Sentence: {text}
JSON:
"""

def classify_with_allowed_conf(text, allowed, fewshot_pairs, temperature=0.0):
    few = make_fewshot_block_scoped(fewshot_pairs, allowed, k_total=12)
    prompt = CLASSIFY_SET_PROMPT_CONF.format(
        label_set=_label_block(allowed),
        text_label_examples=few,
        text=text
    )
    raw = chat_text_cached(prompt, temperature=temperature, max_tokens=32)
    lab, conf = parse_label_and_confidence(raw)
    if lab not in allowed:
        lab = None
    # default confidence if missing
    if conf is None:
        conf = 0.5
    return lab, conf

POS_ALLOWED = {"happiness","surprise"}
NEG_ALLOWED = {"sadness","anger","fear","disgust"}

# NEW classifier: rules -> polarity -> scoped pass with confidence threshold tau
def classify_polarity_first_tau(text, fewshot_pairs, tau=0.60):
    # A) quick lexical rules first
    rb = rule_label(text)
    if rb:
        return rb

    # B) polarity gate
    pol = polarity_gate(text)  # positive | negative | neutral
    if pol == "neutral":
        return "neutral"
    allowed = POS_ALLOWED if pol == "positive" else NEG_ALLOWED

    # C) deterministic scoped pass, get (label, confidence)
    lab, conf = classify_with_allowed_conf(text, allowed, fewshot_pairs, temperature=0.0)

    # If high confidence, accept immediately
    if lab and conf >= tau:
        return lab

    # If low confidence AND no overt emotional cues -> neutral
    if (not has_emotion_cue(text)) and (conf < tau):
        return "neutral"

    # D) small scoped ensemble to stabilize low-conf cases
    ballots = []
    for t in (0.2, 0.35, 0.5):
        lab_k, conf_k = classify_with_allowed_conf(text, allowed, fewshot_pairs, temperature=t)
        if lab_k:
            # You can weight by confidence if you like; simple vote is fine:
            ballots.append(lab_k)

    if ballots:
        return Counter(ballots).most_common(1)[0][0]

    # E) fallback conservative
    return "neutral"

In [29]:
# ===== Grid search tau on dev to maximize Macro-F1 =====
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix

taus = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
scores = []

def pred_with_tau(s, tau):
    return classify_polarity_first_tau(s, fewshot_pairs, tau=tau)

for tau in taus:
    y_true = df_dev["Emotion"].to_numpy()
    y_pred = df_dev["Sentence"].apply(lambda s: pred_with_tau(s, tau)).to_numpy()
    macro = f1_score(y_true, y_pred, average="macro")
    wght  = f1_score(y_true, y_pred, average="weighted")
    scores.append((tau, macro, wght))
    print(f"tau={tau:.2f} -> Macro-F1={macro:.3f} | Weighted-F1={wght:.3f}")

# pick best tau by Macro-F1 (you can switch to weighted if you prefer)
best_tau, best_macro, best_w = max(scores, key=lambda x: x[1])
print(f"\nBest tau on DEV: {best_tau:.2f} (Macro-F1={best_macro:.3f}, Weighted-F1={best_w:.3f})")

# Optional: show a full dev report at the chosen tau
y_true = df_dev["Emotion"].to_numpy()
y_pred = df_dev["Sentence"].apply(lambda s: pred_with_tau(s, best_tau)).to_numpy()
print("\nDEV Report @ best tau:")
print(classification_report(y_true, y_pred, digits=3))

tau=0.40 -> Macro-F1=0.481 | Weighted-F1=0.580
tau=0.50 -> Macro-F1=0.481 | Weighted-F1=0.580
tau=0.55 -> Macro-F1=0.481 | Weighted-F1=0.580
tau=0.60 -> Macro-F1=0.481 | Weighted-F1=0.580
tau=0.65 -> Macro-F1=0.463 | Weighted-F1=0.555
tau=0.70 -> Macro-F1=0.463 | Weighted-F1=0.555
tau=0.75 -> Macro-F1=0.437 | Weighted-F1=0.513
tau=0.80 -> Macro-F1=0.437 | Weighted-F1=0.513

Best tau on DEV: 0.40 (Macro-F1=0.481, Weighted-F1=0.580)

DEV Report @ best tau:
              precision    recall  f1-score   support

       anger      0.615     0.558     0.585        43
     disgust      0.500     0.200     0.286         5
        fear      0.667     0.321     0.433        81
   happiness      0.857     0.720     0.783       125
     neutral      0.498     0.916     0.646       167
     sadness      0.867     0.371     0.520        70
    surprise      0.300     0.073     0.118        41

    accuracy                          0.607       532
   macro avg      0.615     0.451     0.481       532


## 7) Two‑stage prompts (emotive gate → 6‑way classifier)
Strong baseline for macro‑F1 improvements.


In [36]:
CLASSIFY_PROMPT = """Choose exactly ONE label from:
["happiness","sadness","anger","surprise","fear","disgust","neutral"].

Rules:
- Use "neutral" ONLY if the sentence is purely factual/informational with NO affective cues.
- If there is ANY emotional cue (first-person feelings, exclamation/question, intensifiers, value judgement),
  prefer a non-neutral label.
- Output strict JSON only: {{"label":"<one label>"}}.

EXAMPLES:
{text_label_examples}

Sentence: \"\"\"{text}\"\"\""""

CLASSIFY_PROMPT_NO_NEUTRAL = """Choose exactly ONE label from:
["happiness","sadness","anger","surprise","fear","disgust"].
Assume the sentence is NOT neutral; pick the closest label.
Return ONLY JSON: {{"label":"<one label>"}}

EXAMPLES:
{text_label_examples}

Sentence: \"\"\"{text}\"\"\""""
# (braces are escaped with {{ }})


In [37]:
# ===== Combined: two-stage classifier + intensity-aware + disgust guardrail (WEIGHTED ENSEMBLE) =====
from collections import Counter, defaultdict  # <-- added defaultdict

ALLOWED = {"happiness","sadness","anger","surprise","fear","disgust","neutral"}
NON_NEUTRAL = {"happiness","sadness","anger","surprise","fear","disgust"}

def _chat_safe(prompt, **kw):
    """Prefer cache if available; fall back to direct call."""
    try:
        return chat_text_cached(prompt, **kw)
    except NameError:
        return chat_text(prompt, **kw)

def make_fewshot_block(pairs):
    """pairs: list of (text, label) few-shot examples (short, archetypal)."""
    return "\n".join([f'Sentence: "{t}" -> Emotion: {y}' for t, y in pairs])

# Small multi-template set for a conditional ensemble (only used if still uncertain)
CLS_TEMPLATES = [
    CLASSIFY_PROMPT,
    """Task: single-label emotion classification.
Return ONLY JSON: {{\"label\":\"<one label>\"}}.
EXAMPLES:
{text_label_examples}
Sentence: \"\"\"{text}\"\"\""""
]

def _add_intensity_context(prompt: str, intensity: str | None, text: str) -> str:
    if not intensity:
        return prompt
    it = str(intensity).lower()
    hint = (
        f"\nContext: intensity={it}. "
        "If intensity='intense' and the sentence feels negative, prefer anger, fear, or disgust. "
        "If intensity='intense' and the sentence feels positive, prefer happiness or surprise. "
        "If intensity='mild' and there are no strong emotion cues, 'neutral' is permissible."
    )
    return prompt + hint

def classify_two_stage(text, fewshot_pairs, intensity=None, use_ensemble=True, prev="", nxt=""):
    """
    Stage 1: ask with neutral allowed (temp=0.0). If non-neutral, return it.
             If neutral/unparsed and NO emotion cues -> return 'neutral'.
    Stage 2: (only if cues present) ask again forbidding neutral (temp=0.0).
             If still uncertain -> tiny ensemble with weighted vote (+ disgust guardrail).
    """
    # A) High-precision lexical rules first
    rb = rule_label(text)
    if rb:
        return rb

    few = make_fewshot_block(fewshot_pairs)
    pol = guess_polarity(text)  # 'positive' | 'negative' | 'unknown'

    # B) Stage 1 — allow neutral (deterministic)
    p1 = CLASSIFY_PROMPT.format(text=text, text_label_examples=few)
    p1 = _add_intensity_context(p1, intensity, text)
    raw1 = _chat_safe(p1, temperature=0.0, max_tokens=24)
    lab1 = parse_json_or_label(raw1)
    if lab1 in NON_NEUTRAL:
        return lab1

    # If model said neutral (or couldn't parse) and there are no emotion cues -> stop
    # BUT if intensity says 'intense', we force a second look even without cues.
    if (lab1 is None or lab1 == "neutral"):
        if intensity and str(intensity).lower() == "intense":
            pass  # escalate
        elif not has_emotion_cue(text):
            return "neutral"

    # C) Stage 2 — forbid neutral (deterministic)
    p2 = CLASSIFY_PROMPT_NO_NEUTRAL.format(text=text, text_label_examples=few)
    p2 = _add_intensity_context(p2, intensity, text)
    raw2 = _chat_safe(p2, temperature=0.0, max_tokens=24)
    lab2 = parse_json_or_label(raw2)
    if lab2 in NON_NEUTRAL:
        return lab2

    # D) Conditional ensemble (only if still uncertain)
    if use_ensemble:
        ballots = []
        for T in CLS_TEMPLATES:
            for t in (0.2, 0.35, 0.5):  # small diversity only here
                pk_prompt = T.format(text=text, text_label_examples=few)
                pk_prompt = _add_intensity_context(pk_prompt, intensity, text)
                rawk = _chat_safe(pk_prompt, temperature=t, max_tokens=24)
                pk = parse_json_or_label(rawk)
                if pk in ALLOWED:
                    ballots.append(pk)

        # Rare-class guardrail: disgust
        if ballots and "disgust" in ballots and DISGUST_CUE.search(text or ""):
            return "disgust"

        if ballots:
            # === NEW: WEIGHTED VOTE (uses 'weights' computed in your few-shot cell) ===
            sc = defaultdict(float)
            for lab in ballots:
                sc[lab] += weights.get(lab, 1.0)   # weight rare classes higher; de-emphasize neutral

            # optional: small intensity-aware nudge
            if intensity and str(intensity).lower() == "intense":
                if pol == "negative":
                    preferred = ("anger","fear","disgust")
                elif pol == "positive":
                    preferred = ("happiness","surprise")
                else:
                    preferred = tuple()
                for lab in preferred:
                    if lab in sc:
                        sc[lab] += 0.15  # tiny boost

            # pick max-scoring label
            top_label = max(sc.items(), key=lambda kv: kv[1])[0]

            # If 'neutral' wins, prefer best-scoring non-neutral if present
            if top_label == "neutral":
                nn = [(l, s) for l, s in sc.items() if l in NON_NEUTRAL]
                if nn:
                    top_label = max(nn, key=lambda kv: kv[1])[0]
            return top_label

    # E) Final fallback
    return "neutral"

# Backward-compat alias (your evals use this name)
def classify_neutral_last(text, fewshot_pairs, intensity=None):
    return classify_two_stage(text, fewshot_pairs, intensity=intensity, use_ensemble=True)

In [38]:
def evaluate_with_context(name, df_subset, predict_fn):
    s = df_subset["Sentence"].astype(str).tolist()
    it = df_subset["Intensity"].astype(str).tolist() if "Intensity" in df_subset else [None]*len(s)
    prevs = [""] + s[:-1]
    nexts = s[1:] + [""]
    y_true = df_subset["Emotion"].to_numpy()
    y_pred = [
        predict_fn(s[i], it[i], prevs[i], nexts[i])
        for i in range(len(s))
    ]
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

## 8) Evaluate on dev (sample → full), then test

In [39]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import numpy as np
import pandas as pd

LABELS = ["happiness","sadness","anger","surprise","fear","disgust","neutral"]

# --- A) sentence-only evaluator (for your older baselines like polarity-first) ---
def evaluate_sentence_only(name, df_subset, predict_fn, show_cm=False):
    y_true = df_subset["Emotion"].to_numpy()
    y_pred = df_subset["Sentence"].apply(predict_fn).to_numpy()

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))
    macro = f1_score(y_true, y_pred, average="macro")
    weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"Macro-F1: {macro:.3f} | Weighted-F1: {weighted:.3f}")

    if show_cm:
        cm = confusion_matrix(y_true, y_pred, labels=LABELS)
        cm_df = pd.DataFrame(cm, index=[f"T:{l}" for l in LABELS], columns=[f"P:{l}" for l in LABELS])
        print("\nConfusion Matrix:")
        print(cm_df)

    return macro, weighted, y_pred

# --- B) intensity-aware evaluator (row-wise so we can pass Intensity) ---
def evaluate_with_intensity(name, df_subset, predict_fn, show_cm=False):
    y_true = df_subset["Emotion"].to_numpy()
    # pass both sentence and intensity into your classifier
    y_pred = df_subset.apply(lambda r: predict_fn(r["Sentence"], r.get("Intensity")), axis=1).to_numpy()

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))
    macro = f1_score(y_true, y_pred, average="macro")
    weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"Macro-F1: {macro:.3f} | Weighted-F1: {weighted:.3f}")

    if show_cm:
        cm = confusion_matrix(y_true, y_pred, labels=LABELS)
        cm_df = pd.DataFrame(cm, index=[f"T:{l}" for l in LABELS], columns=[f"P:{l}" for l in LABELS])
        print("\nConfusion Matrix:")
        print(cm_df)

    return macro, weighted, y_pred

# fix the dev sample so all methods compare on the same subset
dev_sample = df_dev.sample(80, random_state=7)

# =========================
# INTENSITY-AWARE METHODS
# =========================
# Use this for your new classifier (classify_neutral_last/classify_two_stage that accepts intensity)
macro_ns_dev_s, w_ns_dev_s, _ = evaluate_with_intensity(
    "Neutral-last + rules + intensity-aware ensemble (dev sample)",
    dev_sample,
    lambda s, it: classify_neutral_last(s, fewshot_pairs, intensity=it)
)

macro_ns_dev, w_ns_dev, _ = evaluate_with_intensity(
    "Neutral-last + rules + intensity-aware ensemble (dev full)",
    df_dev,
    lambda s, it: classify_neutral_last(s, fewshot_pairs, intensity=it)
)

macro_ns_te, w_ns_te, y_pred_ns_te = evaluate_with_intensity(
    "FINAL — Neutral-last + rules + intensity-aware ensemble (test)",
    df_test,
    lambda s, it: classify_neutral_last(s, fewshot_pairs, intensity=it),
    show_cm=True
)

# (Optional) also evaluate the two-stage name directly if you want it in the table:
macro_ts_dev, w_ts_dev, _ = evaluate_with_intensity(
    "Two-stage (intensity-aware) (dev full)",
    df_dev,
    lambda s, it: classify_two_stage(s, fewshot_pairs, intensity=it)
)
macro_ts_te, w_ts_te, _ = evaluate_with_intensity(
    "FINAL — Two-stage (intensity-aware) (test)",
    df_test,
    lambda s, it: classify_two_stage(s, fewshot_pairs, intensity=it)
)

# =========================
# SENTENCE-ONLY BASELINES
# =========================
# Keep your polarity-first baseline exactly as you had it (no intensity arg)
# (Comment these two out if you didn't run the tuning cell that defines best_tau.)
macro_pf_dev, w_pf_dev, y_pred_dev = evaluate_sentence_only(
    f"Polarity-first + confidence τ={best_tau:.2f} (dev full)",
    df_dev,
    lambda s: classify_polarity_first_tau(s, fewshot_pairs, tau=best_tau),
    show_cm=True
)

macro_pf_te, w_pf_te, y_pred_pf_te = evaluate_sentence_only(
    f"FINAL — Polarity-first + confidence τ={best_tau:.2f} (test)",
    df_test,
    lambda s: classify_polarity_first_tau(s, fewshot_pairs, tau=best_tau),
    show_cm=True
)

# =========================
# Save predictions (pick which run you want to log — here we save the intensity-aware final)
# =========================
pd.DataFrame({
    "Sentence": df_test["Sentence"].to_numpy(),
    "Gold": df_test["Emotion"].to_numpy(),
    "Pred": y_pred_ns_te
}).to_csv("predictions_test_intensity_aware.csv", index=False)

evaluate_with_context(
    "Two-stage (intensity + context)",
    df_test,
    lambda s, it, p, n: classify_two_stage(s, fewshot_pairs, intensity=it, prev=p, nxt=n)
)

print("\nSaved: predictions_test_intensity_aware.csv")


=== Neutral-last + rules + intensity-aware ensemble (dev sample) ===
              precision    recall  f1-score   support

       anger      0.333     0.750     0.462         4
        fear      0.615     0.667     0.640        12
   happiness      0.500     0.200     0.286        15
     neutral      0.615     0.800     0.696        30
     sadness      0.800     0.500     0.615         8
    surprise      0.750     0.545     0.632        11

    accuracy                          0.600        80
   macro avg      0.602     0.577     0.555        80
weighted avg      0.617     0.600     0.582        80

Macro-F1: 0.555 | Weighted-F1: 0.582

=== Neutral-last + rules + intensity-aware ensemble (dev full) ===
              precision    recall  f1-score   support

       anger      0.524     0.767     0.623        43
     disgust      0.667     0.800     0.727         5
        fear      0.636     0.605     0.620        81
   happiness      0.770     0.456     0.573       125
     neutra

# 9) Error analysis, full run, and logging

In [41]:
# === Error analysis & exports ===
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import pandas as pd

# 1) Predict on the FULL dataset (intensity-aware)
best_fn = lambda s, it: classify_neutral_last(s, fewshot_pairs, intensity=it)
df_all = df.copy()
df_all["Predicted"] = df_all.apply(lambda r: best_fn(r["Sentence"], r.get("Intensity")), axis=1)

if "Predicted" in df_all.columns:
    hard = df_all[(df_all["Emotion"] != "neutral") & (df_all["Predicted"] == "neutral")].copy()
    hard["len"] = hard["Sentence"].str.len()
    hard = hard[(hard["len"] >= 4) & (hard["len"] <= 80)]
    extras = []
    for cls in ["sadness","fear","surprise","happiness","anger","disgust"]:
        samp = (hard[hard["Emotion"] == cls]
                .drop_duplicates(subset=["Sentence"])
                .head(6))
        extras += list(zip(samp["Sentence"], samp["Emotion"]))
    print(f"Adding {len(extras)} mined few-shots.")
    fewshot_pairs = fewshot_pairs + extras

# 2) Overall metrics (context first)
print(classification_report(df_all["Emotion"], df_all["Predicted"], digits=3, zero_division=0))
print("Macro-F1:", f1_score(df_all["Emotion"], df_all["Predicted"], average="macro"))

# 3) Targeted slices (YOUR TWO FILTERS GO HERE)
# false neutrals: gold is non-neutral, pred is neutral
false_neutrals = df_all[(df_all["Emotion"] != "neutral") & (df_all["Predicted"] == "neutral")]
print(f"\nFalse neutrals (non-neutral → neutral): {len(false_neutrals)}")
display(false_neutrals[["Sentence","Emotion","Predicted","Intensity"]].head(12))

# happiness vs surprise boundary (gold happiness, predicted surprise)
happy_as_surprise = df_all[(df_all["Emotion"] == "happiness") & (df_all["Predicted"] == "surprise")]
print(f"\nHappiness → Surprise: {len(happy_as_surprise)}")
display(happy_as_surprise[["Sentence","Emotion","Predicted","Intensity"]].head(12))

# (optional: add the reverse slice too, often insightful)
surprise_as_happy = df_all[(df_all["Emotion"] == "surprise") & (df_all["Predicted"] == "happiness")]
print(f"\nSurprise → Happiness: {len(surprise_as_happy)}")
display(surprise_as_happy[["Sentence","Emotion","Predicted","Intensity"]].head(12))

# 4) Save full predictions + slices for your log/report
df_all.to_csv("prompt_engineering_full_predictions.csv", index=False)
false_neutrals.to_csv("slice_false_neutrals.csv", index=False)
happy_as_surprise.to_csv("slice_happy_as_surprise.csv", index=False)
surprise_as_happy.to_csv("slice_surprise_as_happy.csv", index=False)
print("\nSaved: prompt_engineering_full_predictions.csv and slice CSVs.")

# 5) (optional) quick confusion table to see dominant confusions
ct = pd.crosstab(df_all["Emotion"], df_all["Predicted"], normalize="index").round(3)
print("\nRow-normalized confusion (top 10 rows by size):")
display(ct.reindex(df_all["Emotion"].value_counts().index).head(10))

# 6) Prompt log (same as before)
import csv, time
log_row = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL,
    "approach": "neutral-last + rules + intensity-aware ensemble",
    "fewshot_per_class": len(fewshot_pairs)//7,
    "notes": "curated+top-up few-shots; two-stage; intensity hint; caching on"
}
with open("prompt_log.csv", "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=log_row.keys())
    if f.tell() == 0: w.writeheader()
    w.writerow(log_row)
print("Logged to prompt_log.csv")

Adding 25 mined few-shots.
              precision    recall  f1-score   support

       anger      0.514     0.704     0.594        54
     disgust      0.800     0.667     0.727         6
        fear      0.561     0.772     0.650       101
   happiness      0.827     0.519     0.638       156
     neutral      0.617     0.780     0.689       209
     sadness      0.811     0.341     0.480        88
    surprise      0.562     0.529     0.545        51

    accuracy                          0.633       665
   macro avg      0.670     0.616     0.618       665
weighted avg      0.673     0.633     0.625       665

Macro-F1: 0.6176414724718607

False neutrals (non-neutral → neutral): 101


,Sentence,Emotion,Predicted,Intensity
0,"Previously on Australian Survivor, Tara and Lo...",happiness,neutral,mild
14,Go!,anger,neutral,moderate
46,It was Tara who spilled the beans to me.,surprise,neutral,mild
58,you don't know this Tara.,anger,neutral,mild
62,"She spilled the beans to me, like, around the ...",surprise,neutral,mild
106,and move and develop in Survivor.,happiness,neutral,mild
109,No.,anger,neutral,mild
111,I got Ziggy on board with the lollies because ...,happiness,neutral,mild
113,then maybe she might trust me later on down th...,happiness,neutral,mild
116,Yesterday I had this beautiful plan to get Lac...,happiness,neutral,mild



Happiness → Surprise: 3


,Sentence,Emotion,Predicted,Intensity
280,You guys ready to get to today's immunity chal...,happiness,surprise,mild
325,"If Lockie takes it today, it will be an incred...",happiness,surprise,moderate
426,Go three three and tie it up,happiness,surprise,moderate



Surprise → Happiness: 0


,Sentence,Emotion,Predicted,Intensity



Saved: prompt_engineering_full_predictions.csv and slice CSVs.

Row-normalized confusion (top 10 rows by size):


Predicted,anger,disgust,fear,happiness,neutral,sadness,surprise
Emotion,,,,,,,
neutral,0.038,0.000,0.062,0.053,0.780,0.024,0.043
happiness,0.045,0.006,0.090,0.519,0.314,0.006,0.019
fear,0.109,0.000,0.772,0.010,0.089,0.000,0.020
sadness,0.068,0.000,0.239,0.045,0.250,0.341,0.057
anger,0.704,0.000,0.167,0.019,0.056,0.019,0.037
surprise,0.059,0.000,0.078,0.000,0.333,0.000,0.529
disgust,0.167,0.667,0.000,0.000,0.167,0.000,0.000


Logged to prompt_log.csv


In [35]:
# Strict zero-shot baseline (full dev/test)
BASELINE_PROMPT_JSON = """You must choose exactly one label from:
["happiness","sadness","anger","surprise","fear","disgust","neutral"].
Return ONLY JSON: {{"label":"<one>"}}.

Sentence: \"\"\"{text}\"\"\""""

def classify_baseline_strict(text):
    raw = chat_text_cached(BASELINE_PROMPT_JSON.format(text=text), temperature=0.0, max_tokens=20)
    lab = parse_json_or_label(raw)
    return lab if lab in ALLOWED else "neutral"

# Evaluate apples-to-apples
_ = evaluate_sentence_only("Zero-shot JSON baseline (dev full)",
                           df_dev, classify_baseline_strict, show_cm=True)

_ = evaluate_sentence_only("FINAL — Zero-shot JSON baseline (test)",
                           df_test, classify_baseline_strict, show_cm=True)


=== Zero-shot JSON baseline (dev full) ===
              precision    recall  f1-score   support

       anger      0.492     0.674     0.569        43
     disgust      0.500     0.600     0.545         5
        fear      0.732     0.506     0.599        81
   happiness      0.894     0.608     0.724       125
     neutral      0.574     0.886     0.696       167
     sadness      0.760     0.543     0.633        70
    surprise      0.778     0.341     0.475        41

    accuracy                          0.656       532
   macro avg      0.676     0.594     0.606       532
weighted avg      0.706     0.656     0.651       532

Macro-F1: 0.606 | Weighted-F1: 0.651

Confusion Matrix:
             P:happiness  P:sadness  P:anger  P:surprise  P:fear  P:disgust  \
T:happiness           76          0        3           1       2          1   
T:sadness              2         38        4           1       7          1   
T:anger                1          3       29           1       3  